# Market Microstructure Research: Avellaneda-Stoikov Market Making

This notebook walks through the core simulation, analytics, and key results.

**Structure:**
1. Simulation setup and parameter choices
2. Core market dynamics (price, spread, regime)
3. MM inventory evolution and risk metrics
4. PnL decomposition (spread capture vs adverse selection)
5. OFI predictability analysis
6. Parameter sensitivity: γ, σ, informed fraction
7. Break-even conditions

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from src.core.simulator import MarketSimulator, SimConfig, RegimeParams
from src.strategies.avellaneda_stoikov import ASParams, AvellanedaStoikov
from src.analytics.analytics import full_report, pnl_decomposition, ofi_predictability
from src.analytics.features import build_feature_matrix

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
COLORS = sns.color_palette('tab10')
print('Imports OK')

## 1. Baseline Simulation

In [ ]:
cfg = SimConfig(
    n_steps=5000,
    seed=42,
    mm_params=ASParams(gamma=0.1, sigma=0.002, kappa=1.5),
    n_noise_traders=15,
    n_informed_traders=3,
)
sim = MarketSimulator(cfg)
df = sim.run()
print(f'Simulation complete: {len(df)} steps')
print(f'Price: {df.mid_price.iloc[0]:.2f} → {df.mid_price.iloc[-1]:.2f}')
df.head(3)

## 2. Core Market Dynamics

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Mid price with regime colouring
ax = axes[0]
low_mask = df.regime_name == 'low_vol'
high_mask = df.regime_name == 'high_vol'
ax.plot(df.step[low_mask], df.mid_price[low_mask], color=COLORS[0], lw=0.8, label='Low vol')
ax.plot(df.step[high_mask], df.mid_price[high_mask], color=COLORS[1], lw=0.8, label='High vol')
ax.set_ylabel('Mid price')
ax.legend(fontsize=9)
ax.set_title('Mid Price by Volatility Regime')

# Spread
ax = axes[1]
ax.plot(df.step, df.spread, color=COLORS[2], lw=0.6, alpha=0.8)
ax.axhline(df.spread.mean(), color='red', ls='--', lw=1, label=f'Mean={df.spread.mean():.4f}')
ax.set_ylabel('Spread')
ax.set_title('Quoted Spread')
ax.legend(fontsize=9)

# OFI
ax = axes[2]
ax.plot(df.step, df.ofi.rolling(20).mean(), color=COLORS[3], lw=0.8)
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_ylabel('OFI (20-step MA)')
ax.set_title('Order Flow Imbalance')

# Trade count
ax = axes[3]
ax.plot(df.step, df.total_trades.diff().fillna(0).rolling(20).mean(), color=COLORS[4], lw=0.8)
ax.set_ylabel('Trade rate')
ax.set_xlabel('Step')
ax.set_title('Trade Rate (20-step MA)')

plt.tight_layout()
plt.savefig('../docs/figures/market_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. MM Inventory and PnL

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Inventory
ax = axes[0]
ax.fill_between(df.step, df.mm_inventory, alpha=0.5, color=COLORS[0])
ax.axhline(0, color='black', lw=0.5)
ax.axhline(cfg.mm_params.max_inventory, color='red', ls='--', lw=1, label=f'Limit ±{cfg.mm_params.max_inventory}')
ax.axhline(-cfg.mm_params.max_inventory, color='red', ls='--', lw=1)
ax.set_ylabel('Inventory (units)')
ax.set_title('MM Inventory Path')
ax.legend(fontsize=9)

# Total PnL
ax = axes[1]
ax.plot(df.step, df.mm_total_pnl, color=COLORS[2], lw=0.8)
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Total PnL')
ax.set_title('MM Total PnL (cash + mark-to-market)')

# PnL decomposition
ax = axes[2]
ax.plot(df.step, df.mm_spread_capture, color=COLORS[3], lw=0.8, label='Spread capture')
ax.plot(df.step, df.mm_adverse_selection, color=COLORS[1], lw=0.8, label='Adverse selection')
net = df.mm_spread_capture - df.mm_adverse_selection
ax.plot(df.step, net, color=COLORS[0], lw=1.2, label='Net alpha')
ax.set_ylabel('Cumulative')
ax.set_xlabel('Step')
ax.set_title('PnL Components')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../docs/figures/inventory_pnl.png', dpi=150, bbox_inches='tight')
plt.show()

rpt = full_report(df)
print('\nPnL Decomposition:')
for k, v in rpt['pnl'].items():
    if isinstance(v, float):
        print(f'  {k:30s}: {v:10.4f}')

## 4. A-S Model: Spread and Reservation Price

In [ ]:
# Illustrate how reservation price and spread change with inventory
import math

mm_model = AvellanedaStoikov(ASParams(gamma=0.1, sigma=0.01, kappa=1.5, T=1.0))
mid = 100.0
inventories = range(-30, 31)
t = 0.5

res_prices = [mm_model.reservation_price(mid, q, t) for q in inventories]
bids = [mm_model.compute_quotes(mid, q, t)[0] for q in inventories]
asks = [mm_model.compute_quotes(mid, q, t)[1] for q in inventories]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(list(inventories), res_prices, label='Reservation price', color=COLORS[2], lw=2)
ax.plot(list(inventories), bids, label='Bid quote', color=COLORS[0], lw=1.5, ls='--')
ax.plot(list(inventories), asks, label='Ask quote', color=COLORS[1], lw=1.5, ls='--')
ax.axhline(mid, color='black', lw=0.8, ls=':', label='Mid price')
ax.set_xlabel('Inventory (units)')
ax.set_ylabel('Price')
ax.set_title('A-S Quotes vs Inventory\n(γ=0.1, σ=0.01, t=0.5)')
ax.legend(fontsize=9)

# Fill probability curve
ax = axes[1]
deltas = np.linspace(0, 1.0, 100)
kappas = [0.5, 1.0, 1.5, 2.5]
for kappa in kappas:
    probs = [math.exp(-kappa * d) for d in deltas]
    ax.plot(deltas, probs, label=f'κ={kappa}', lw=1.5)
ax.set_xlabel('Half-spread δ')
ax.set_ylabel('Fill probability')
ax.set_title('Fill Probability P(fill|δ) = exp(-κδ)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../docs/figures/as_model.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. OFI Predictability

In [ ]:
horizons = [1, 2, 3, 5, 10, 15, 20, 30, 50]
ics, betas, r2s = [], [], []

for h in horizons:
    res = ofi_predictability(df, horizon=h)
    ics.append(res.get('information_coefficient', 0))
    betas.append(res.get('beta', 0))
    r2s.append(res.get('r_squared', 0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.bar(range(len(horizons)), ics, color=COLORS[0], alpha=0.8)
ax.set_xticks(range(len(horizons)))
ax.set_xticklabels([str(h) for h in horizons])
ax.set_xlabel('Horizon (steps)')
ax.set_ylabel('Information Coefficient (Spearman ρ)')
ax.set_title('OFI Signal Decay with Horizon')
ax.axhline(0, color='black', lw=0.8)

ax = axes[1]
ax.bar(range(len(horizons)), r2s, color=COLORS[2], alpha=0.8)
ax.set_xticks(range(len(horizons)))
ax.set_xticklabels([str(h) for h in horizons])
ax.set_xlabel('Horizon (steps)')
ax.set_ylabel('R²')
ax.set_title('OFI Regression R² by Horizon')

plt.tight_layout()
plt.savefig('../docs/figures/ofi_decay.png', dpi=150, bbox_inches='tight')
plt.show()

print('IC by horizon:')
for h, ic in zip(horizons, ics):
    print(f'  h={h:3d}: IC={ic:.4f}')

## 6. Volatility vs Spread (Experiment 1)

In [ ]:
sigmas = [0.001, 0.002, 0.004, 0.008, 0.015, 0.025]
results = []

for sigma in sigmas:
    regimes = [
        RegimeParams('main', 0.0, sigma, 0.01, sigma*0.5),
        RegimeParams('spike', 0.0, sigma*3, 0.05, sigma),
    ]
    cfg_s = SimConfig(n_steps=2000, regimes=regimes, mm_params=ASParams(sigma=sigma), seed=1)
    df_s = MarketSimulator(cfg_s).run()
    results.append({
        'sigma': sigma,
        'mean_spread': df_s.spread.mean(),
        'pnl': df_s.mm_total_pnl.iloc[-1],
        'spread_capture': df_s.mm_spread_capture.iloc[-1],
    })

res_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(res_df.sigma, res_df.mean_spread, s=80, color=COLORS[0])
ax.plot(res_df.sigma, res_df.mean_spread, color=COLORS[0], lw=1.5, ls='--')
ax.set_xlabel('Volatility σ (per step)')
ax.set_ylabel('Mean Quoted Spread')
ax.set_title('Volatility vs Mean Spread\n(A-S prediction: linear in σ²)')

ax = axes[1]
ax.scatter(res_df.sigma**2, res_df.mean_spread, s=80, color=COLORS[2])
slope, intercept, r, p, se = stats.linregress(res_df.sigma**2, res_df.mean_spread)
x_fit = np.linspace(0, res_df.sigma.max()**2, 50)
ax.plot(x_fit, intercept + slope*x_fit, color='red', lw=1.5, ls='--',
        label=f'OLS fit: R²={r**2:.3f}')
ax.set_xlabel('Variance σ² (per step)')
ax.set_ylabel('Mean Quoted Spread')
ax.set_title('Spread vs Variance (test linearity)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../docs/figures/vol_vs_spread.png', dpi=150, bbox_inches='tight')
plt.show()
print(res_df)

## 7. Informed Fraction vs MM Net Alpha (Experiment 5)

In [ ]:
n_informed_range = [0, 1, 2, 3, 5, 7, 10]
total_participants = 18
alpha_results = []

for n_inf in n_informed_range:
    cfg_i = SimConfig(n_steps=3000, n_informed_traders=n_inf, n_noise_traders=15, seed=5)
    df_i = MarketSimulator(cfg_i).run()
    pnl_d = pnl_decomposition(df_i)
    alpha_results.append({
        'n_informed': n_inf,
        'informed_pct': n_inf / total_participants * 100,
        'net_alpha': pnl_d.get('net_alpha', 0),
        'spread_capture': pnl_d.get('spread_capture', 0),
        'adverse_selection': pnl_d.get('adverse_selection_loss', 0),
    })

ar = pd.DataFrame(alpha_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ar.informed_pct, ar.spread_capture, label='Spread capture', color=COLORS[0], alpha=0.8)
ax.bar(ar.informed_pct, -ar.adverse_selection, label='Adverse selection loss', color=COLORS[1], alpha=0.8)
ax.plot(ar.informed_pct, ar.net_alpha, 'k-o', lw=2, label='Net alpha', markersize=6)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Informed fraction (% of participants)')
ax.set_ylabel('Cumulative PnL component')
ax.set_title('MM Net Alpha vs Informed Trading Fraction')
ax.legend(fontsize=10)

# Find break-even
crossover = ar[ar.net_alpha < 0].informed_pct.min()
if not pd.isna(crossover):
    ax.axvline(crossover, color='red', ls='--', lw=1.5, label=f'Break-even ≈{crossover:.0f}%')
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../docs/figures/informed_fraction.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Importance Analysis

In [ ]:
features = build_feature_matrix(df, include_targets=True)

from src.analytics.features import feature_importance_ols
importance = feature_importance_ols(features, 'target_ret_5').head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [COLORS[0] if v > 0 else COLORS[1] for v in importance.values]
bars = ax.barh(range(len(importance)), importance.values, color=colors, alpha=0.8)
ax.set_yticks(range(len(importance)))
ax.set_yticklabels(importance.index, fontsize=9)
ax.set_xlabel('t-statistic (OLS)')
ax.set_title('Feature Importance for 5-Step Forward Return\n(by t-statistic, blue=positive, red=negative)')
ax.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.savefig('../docs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nFeature matrix shape: {features.shape}')
print(f'Top feature: {importance.index[0]} (t={importance.iloc[0]:.2f})')